# GAN Defect Augmentation - Phase 2: GAN Training

This notebook guides you through training the WGAN-GP for defect generation.

## Step 1: Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn as nn
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

from src.utils.config import load_config, create_directories
from src.utils.logger import setup_logger
from src.data.mvtec_dataset import MVTecDataLoader
from src.models.generator import Generator
from src.models.discriminator import Discriminator

# Setup
config = load_config('../config.yaml')
create_directories(config)
logger = setup_logger('gan-notebook', config.training.log_dir)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
print(f"CUDA available: {torch.cuda.is_available()}")

## Step 2: Initialize Models

In [ ]:
# Create models
generator = Generator(
    input_channels=6,
    output_channels=3,
    num_classes=len(config.data.categories),
    defect_embedding_dim=config.gan.defect_embedding_dim,
).to(device)

discriminator = Discriminator(
    input_channels=4,
    base_channels=64,
    num_scales=3,
).to(device)

# Count parameters
gen_params = sum(p.numel() for p in generator.parameters())
disc_params = sum(p.numel() for p in discriminator.parameters())

print(f"Generator parameters: {gen_params:,}")
print(f"Discriminator parameters: {disc_params:,}")
print(f"Total parameters: {gen_params + disc_params:,}")

## Step 3: Load Data

In [ ]:
# Load data for bottle category
category = 'bottle'
data_loader = MVTecDataLoader(
    root_dir=config.data.raw_dir,
    category=category,
    batch_size=config.data.batch_size,
    num_workers=0,  # Set to 0 for notebook
    image_size=config.data.image_size,
)

train_loader, val_loader, test_loader = data_loader.get_loaders()

print(f"Category: {category}")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

## Step 4: Test Forward Pass

In [ ]:
# Get a batch
batch = next(iter(train_loader))

real_images = batch['image'].to(device)
defect_masks = batch['mask'].unsqueeze(1).to(device)
defect_types = batch['label'].to(device)

print(f"Real images shape: {real_images.shape}")
print(f"Defect masks shape: {defect_masks.shape}")
print(f"Defect types shape: {defect_types.shape}")

# Test generator
with torch.no_grad():
    fake_images = generator(real_images, defect_masks, defect_types)
    print(f"Generated images shape: {fake_images.shape}")
    
    # Test discriminator
    d_real = discriminator(real_images, defect_masks)
    d_fake = discriminator(fake_images, defect_masks)
    
    print(f"Discriminator real output shape: {d_real.shape}")
    print(f"Discriminator fake output shape: {d_fake.shape}")

## Step 5: Visualize Generated Images

In [ ]:
# Denormalize for visualization
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

fig, axes = plt.subplots(4, 3, figsize=(12, 12))

for i in range(4):
    # Real image
    real_img = real_images[i].cpu().permute(1, 2, 0).numpy()
    real_img = (real_img * std + mean).clip(0, 1)
    axes[i, 0].imshow(real_img)
    axes[i, 0].set_title(f"Real {i}")
    axes[i, 0].axis('off')
    
    # Defect mask
    mask = defect_masks[i, 0].cpu().numpy()
    axes[i, 1].imshow(mask, cmap='gray')
    axes[i, 1].set_title(f"Mask {i}")
    axes[i, 1].axis('off')
    
    # Generated image
    fake_img = fake_images[i].cpu().detach().permute(1, 2, 0).numpy()
    fake_img = (fake_img * std + mean).clip(0, 1)
    axes[i, 2].imshow(fake_img)
    axes[i, 2].set_title(f"Generated {i}")
    axes[i, 2].axis('off')

plt.tight_layout()
plt.savefig('../outputs/gan_test_generation.png', dpi=150, bbox_inches='tight')
plt.show()

print("Test generation visualization saved")

## Step 6: Training Configuration

In [ ]:
# For quick testing, reduce epochs
test_epochs = 5  # Change to config.gan.epochs for full training

print(f"Training configuration:")
print(f"  Epochs: {test_epochs}")
print(f"  Batch size: {config.data.batch_size}")
print(f"  Learning rate G: {config.gan.learning_rate_g}")
print(f"  Learning rate D: {config.gan.learning_rate_d}")
print(f"  Gradient penalty weight: {config.gan.gradient_penalty_weight}")
print(f"  Discriminator steps: {config.gan.discriminator_steps}")

## Step 7: Run Full Training

For full training, run from command line:

```bash
python src/train_gan.py --config config.yaml
```

Expected training time: 2-3 days on RTX 4090

## Next Steps

After training completes:

1. Check checkpoints in `checkpoints/`
2. Monitor training with wandb
3. Proceed to Phase 3: Quality Control

```bash
python src/evaluate_quality.py --config config.yaml
```